In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import lightgbm
from lightgbm import LGBMClassifier
from tree_influence.explainers import BoostIn
import pandas as pd 
import sklearn 
import re
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.model_selection import GridSearchCV

In [ ]:
# load german credit data 
data = pd.read_csv('german_credit_data.csv', index_col=0)

# encode binary value 
data["Risk"] = data["Risk"].map({'good': 0, 'bad': 1}).astype(int) #small code adjustement with map to avoid errors

# check counts 
data['Risk'].value_counts(dropna=False)

In [ ]:
data.head()

In [ ]:
# categorical columns have type 'object', should transform in category pandas dtype for LightGBM
cat_cols = ["Sex", "Housing", "Saving accounts", "Checking account", "Purpose"]
for c in cat_cols: 
    data[c] = data[c].astype('category')

# check 
print(data.dtypes)

In [ ]:
y = data['Risk']
X = pd.get_dummies(data.drop(columns="Risk"), drop_first=True)  # everything except risk + categorical in dummies

# clean feature names
X.columns = [
    re.sub(r"\s+", "_", str(col)) for col in X.columns
]

# train/ test split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y) # stratify=y pour garder la même proportion de bon et mauvais 

# convert to numpy -- why? 
X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

## Training

In [ ]:
# train GBDT model
#define model and its hyperparameters here in case we would like to change them later 
#grid search to find the best hyperparameters 
param_grid = { 
    "n_estimators": [25, 50, 100], 
    "num_leaves": [5, 7, 10, 15],
    "learning_rate": [0.05, 0.1]
}

base_model = LGBMClassifier(
    random_state=42,
    verbose=-1,
    verbosity=-1,
    force_col_wise=True
)

grid = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    refit=True,
    verbose=0,
    n_jobs=1
)

#fit best model
grid.fit(X_train, y_train)
fitted_model = grid.best_estimator_
print("Best parameters:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)

In [ ]:
# fit BoostIn influence estimator on the final trained GBDT model
# using the same training data used to train the model
#explainer = BoostIn().fit(fitted_model, X_train_np, y_train_np) # why numpy? 
explainer = BoostIn().fit(fitted_model, X_train, y_train)

# # estimate training influences on each test instance
influence = explainer.get_local_influence(X_test, y_test)  # shape=(no. train, no. test)

# # extract influence values for the first test instance
values = influence[:, 0]  # retrieve first column, meaning influence of each training example on first test

# # sort training examples from:
# # - most positively influential (decreases loss of the test instance the most), to
# # - most negatively influential (increases loss of the test instance the most)
training_idxs = np.argsort(values)[::-1]

## Visualisation of top 5 most influential points for a given test point

In [ ]:
print("Tested client profile :")
print(X_test.iloc[0])
print("True label:", y_test.iloc[0])
print("Model prediction:", model.predict(X_test_np[[0]])[0])
print("Predicted probability:", model.predict_proba(X_test_np[[0]])[0,1])

In [ ]:
print("Influence matrix :", influence.shape)
print("Top 5 proponent examples:", training_idxs[:5])
print("Top 5 oponont examples:", training_idxs[-5:])

# Et regarder les vrais exemples concernés :
data.iloc[training_idxs[:5]]

In [ ]:
# --- 1. Données ---
data = pd.read_csv("german_credit_data.csv", index_col=0, na_values=["NA"])
data["Risk"] = data["Risk"].replace({"good": 0, "bad": 1})
y = data["Risk"]
X = pd.get_dummies(data.drop(columns="Risk"), drop_first=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)
X_tr_np, X_te_np = X_tr.to_numpy(), X_te.to_numpy()
y_tr_np, y_te_np = y_tr.to_numpy(), y_te.to_numpy()

# --- 2. Modèle de base + losses de référence ---
MODEL_PARAMS = dict(n_estimators=100, random_state=1,
                    deterministic=True, verbose=-1)

def per_sample_logloss(model, X, y):
    p = model.predict_proba(X)
    return -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))

base_model = LGBMClassifier(**MODEL_PARAMS).fit(X_tr_np, y_tr_np)
base_losses = per_sample_logloss(base_model, X_te_np, y_te_np)

# --- 3. Choix des 50 tests + BoostIn ---
N_TEST, K = 50, 15
rng = np.random.default_rng(0)
test_idx = rng.choice(len(X_te_np), size=N_TEST, replace=False)

explainer = BoostIn().fit(base_model, X_tr_np, y_tr_np)
influence = explainer.get_local_influence(X_te_np[test_idx], y_te_np[test_idx])
# shape : (n_train, N_TEST)

# --- 4. LOO retraining sur les top-k de chaque test ---
predicted, actual = [], []

for t, te_i in enumerate(test_idx):
    col = influence[:, t]
    topk = np.argsort(np.abs(col))[-K:]              # |influence| → on garde proponents ET opponents

    for tr_i in topk:
        mask = np.ones(len(X_tr_np), dtype=bool)
        mask[tr_i] = False
        m_loo = LGBMClassifier(**MODEL_PARAMS).fit(X_tr_np[mask], y_tr_np[mask])
        loo_loss = per_sample_logloss(m_loo, X_te_np[[te_i]], y_te_np[[te_i]])[0]

        predicted.append(col[tr_i])                  # signe à valider, voir ci-dessous
        actual.append(loo_loss - base_losses[te_i])

predicted = np.array(predicted)
actual = np.array(actual)

# --- 5. Pearson + plot ---
r, p = pearsonr(predicted, actual)
print(f"N = {len(predicted)}   Pearson r = {r:+.3f}   (p = {p:.2e})")

plt.figure(figsize=(6, 6))
plt.scatter(predicted, actual, alpha=0.4, s=18)
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.xlabel("BoostIn-predicted Δ loss on removal")
plt.ylabel("Actual Δ loss (LOO retrain)")
plt.title(f"BoostIn vs LOO — Pearson r = {r:+.3f}")
plt.tight_layout(); plt.show()


# Leave-One-Out Baseline

In [ ]:
import time 
from sklearn.metrics import log_loss 
from sklearn.base import clone

In [ ]:
# --- Original losses for ALL test applicants ---
base_proba = fitted_model.predict_proba(X_test)
classes = pd.Index(fitted_model.classes_)
y_test_array = np.array(y_test)

# position of the true class for each test applicant
true_class_positions = classes.get_indexer(y_test_array)

# probability assigned to the true class
base_true_proba = base_proba[np.arange(len(y_test)), true_class_positions]

# individual log loss for each test applicant
base_losses = -np.log(base_true_proba)

# --- LOO for all test applicants ---
loo_results = []
start = time.time()

for train_idx in X_train.index:
    
    # remove one training example
    X_train_minus = X_train.drop(index=train_idx)
    y_train_minus = y_train.drop(index=train_idx)
    
    # retrain model
    loo_model = clone(fitted_model)
    loo_model.fit(X_train_minus, y_train_minus)
    
    # predict probabilities for ALL test applicants
    proba_minus = loo_model.predict_proba(X_test)
    
    # probability assigned to the true class for each test applicant
    minus_true_proba = proba_minus[np.arange(len(y_test)), true_class_positions]
    
    # individual log losses after removing train_idx
    losses_minus = -np.log(minus_true_proba)
    
    # LOO scores for all test applicants
    loo_scores = losses_minus - base_losses
    
    # store one row
    row = pd.Series(loo_scores, index=X_test.index, name=train_idx)
    loo_results.append(row)

end = time.time()

loo_df = pd.DataFrame(loo_results)

print("LOO runtime:", end - start, "seconds")
print("Shape:", loo_df.shape)
display(loo_df.head())

We can have an idea of what are the most influencial data points for a precise test point by using the below

In [ ]:
target_idx = X_test.index[0]

top_supporters = (
    loo_df[target_idx]
    .sort_values(ascending=False)
    .head(10)
    .rename(f"loo_score_{target_idx}")
    .to_frame()
)

top_opponents = (
    loo_df[target_idx]
    .sort_values(ascending=True)
    .head(10)
    .rename(f"loo_score_{target_idx}")
    .to_frame()
)

top_supporters.index.name = "remove_train_idx"
top_opponents.index.name = "remove_train_idx"

print("Top supporters:")
display(top_supporters)

print("Top opponents:")
display(top_opponents)

In [ ]:
# #Select one test applicant to explain -- test on a single applicant
# target_idx = X_test.index[0]
# x_target = X_test.loc[[target_idx]]
# y_target = y_test.loc[[target_idx]]

# #Original loss on this applicant 
# base_proba = fitted_model.predict_proba(x_target) #predicts the binary probabilities for the two classes 
# base_loss = log_loss(y_target, base_proba, labels=fitted_model.classes_)
# print("Target Applicant: ", target_idx)
# print("True label: ", y_target.iloc[0])
# print("Original loss: ", base_loss)

# ## LOO results 
# loo_results = []
# start = time.time() #to keep track of rntime

# for train_idx in X_train.index: 
#     #remove one training example
#     X_train_minus = X_train.drop(index=train_idx)
#     y_train_minus = y_train.drop(index=train_idx)
    
#     #retrain the GBT model without the train_idx obs
#     loo_model = clone(fitted_model)
#     loo_model.fit(X_train_minus, y_train_minus)
    
#     #compute loss on the same target applicant but with the newly trained model 
#     proba_minus = loo_model.predict_proba(x_target) 
#     loss_minus = log_loss(y_target, proba_minus, labels=loo_model.classes_)
    
#     #LOO influence score 
#     loo_score = loss_minus - base_loss #computes loo_score
#     loo_results.append({
#         "remove_train_idx": train_idx, 
#         f"loo_score_{target_idx}" : loo_score
#     })
    
# end = time.time()

# loo_df = pd.DataFrame(loo_results).set_index("remove_train_idx")
# print("LOO runtime:", end-start, "seconds")
# display(loo_df.head())

# #Training examples that SUPPORT the target prediction 
# top_supporters = loo_df.sort_values("loo_score", ascending = False).head(10)
# #Training examples that HURT the target prediction 
# top_opponent =  loo_df.sort_values("loo_score", ascending = True).head(10)

# print("Top supporters:")
# display(top_supporters)
# print("Top oppponents:")
# display(top_opponent)

# BoostIn custom implementation
Instead of retraining the model like LOO, we look inside the trained GBT and ask: How often does a training applicant follow the same tree paths as the target applicant? To do so, we use BoostIn where we sum influence over trees and a training only contributes at a tree if it lands in the same leaf as the target point. 

In [ ]:
start = time.time()
#1: Get leaf index for every train and test applicant 

leaf_train = fitted_model.predict(X_train, pred_leaf= True)
leaf_test =  fitted_model.predict(X_test, pred_leaf= True)

# print("leaf_train shape:", leaf_train.shape)
# print("leaf_test shape:", leaf_test.shape)


In [ ]:
#2: Count same-leaf matches between each train applicant and each test applicant 
n_train = leaf_train.shape[0]
n_test = leaf_test.shape[0]
n_trees = leaf_train.shape[1]

same_leaf_count = np.zeros((n_train, n_test))

for t in range(n_trees):
    same_leaf_count += (leaf_train[:, [t]] == leaf_test[:, t])


In [ ]:
#3: Compute predicted probabilities
proba_train = fitted_model.predict_proba(X_train)
proba_test = fitted_model.predict_proba(X_test)

# Use the second class as the "positive" class
positive_class = fitted_model.classes_[1]

print("Classes:", fitted_model.classes_)
print("Positive class used:", positive_class)

# Probability of positive class
p_train = proba_train[:, 1]
p_test = proba_test[:, 1]

# Convert true labels to 0/1
y_train_binary = (y_train == positive_class).astype(int).to_numpy()
y_test_binary = (y_test == positive_class).astype(int).to_numpy()

# Gradient of binary log loss with respect to model score
grad_train = p_train - y_train_binary
grad_test = p_test - y_test_binary

In [ ]:
#4 BoostIn style influence score 
boostin_scores = same_leaf_count * (grad_train[:, None] * grad_test[None, :])

boostin_df = pd.DataFrame(
    boostin_scores,
    index=X_train.index,
    columns=X_test.index
)

end = time.time()

print("BoostIn-style runtime:", end - start, "seconds")
print("Shape:", boostin_df.shape)
display(boostin_df.head())

In [ ]:
#5. Compare with LOO

target_idx = X_test.index[0]

loo_top10 = loo_df[target_idx].sort_values(ascending=False).head(10)
boostin_top10 = boostin_df[target_idx].sort_values(ascending=False).head(10)

print("LOO top supporters:")
display(loo_top10.rename("loo_score").to_frame())

print("BoostIn top supporters:")
display(boostin_top10.rename("boostin_score").to_frame())

In [ ]:
#Similarity Check
k = 10

loo_topk = set(
    loo_df[target_idx]
    .sort_values(ascending=False)
    .head(k)
    .index
)

boostin_topk = set(
    boostin_df[target_idx]
    .sort_values(ascending=False)
    .head(k)
    .index
)

overlap = loo_topk.intersection(boostin_topk)

print("LOO top-k:", loo_topk)
print("BoostIn top-k:", boostin_topk)
print("Common influential points:", overlap)
print("Top-k overlap:", len(overlap) / k)

#Spearman correlation
from scipy.stats import spearmanr
rho = spearmanr(
    loo_df[target_idx],
    boostin_df[target_idx]
).correlation

print("Spearman correlation:", rho)

## Overall summary accross indices 

In [ ]:
def topk_overlap(a, b, k=10, ascending=False):
    top_a = set(a.sort_values(ascending=ascending).head(k).index)
    top_b = set(b.sort_values(ascending=ascending).head(k).index)
    return len(top_a & top_b) / k

comparison_results = []

k = 10

for target_idx in X_test.index:
    
    loo_scores = loo_df[target_idx]
    boostin_scores = boostin_df[target_idx]
    
    # Spearman rank correlation
    rho = spearmanr(loo_scores, boostin_scores).correlation
    
    # Top supporters: highest positive influence
    supporter_overlap = topk_overlap(
        loo_scores,
        boostin_scores,
        k=k,
        ascending=False
    )
    
    # Top opponents: most negative influence
    opponent_overlap = topk_overlap(
        loo_scores,
        boostin_scores,
        k=k,
        ascending=True
    )
    
    # Most influential in absolute value
    loo_abs_top = set(loo_scores.abs().sort_values(ascending=False).head(k).index)
    boostin_abs_top = set(boostin_scores.abs().sort_values(ascending=False).head(k).index)
    absolute_overlap = len(loo_abs_top & boostin_abs_top) / k
    
    # Sign agreement
    sign_agreement = np.mean(np.sign(loo_scores) == np.sign(boostin_scores))
    
    comparison_results.append({
        "target_idx": target_idx,
        "spearman": rho,
        f"top{k}_supporter_overlap": supporter_overlap,
        f"top{k}_opponent_overlap": opponent_overlap,
        f"top{k}_absolute_overlap": absolute_overlap,
        "sign_agreement": sign_agreement
    })

comparison_df = pd.DataFrame(comparison_results).set_index("target_idx")
overview = comparison_df.mean(numeric_only=True).to_frame("average_score")
display(overview)
# display(comparison_df.head())
# display(comparison_df.describe())

In [ ]:
display(
    comparison_df.sort_values("spearman", ascending=False).head(10)
)

In [ ]:
display(
    comparison_df.sort_values("spearman", ascending=True).head(10)
)

In [ ]:
import matplotlib.pyplot as plt 
comparison_df["spearman"].hist(bins=20)

In [ ]:
comparison_df[[f"top{k}_supporter_overlap", f"top{k}_opponent_overlap", f"top{k}_absolute_overlap"]].hist(bins=10)

In [ ]:
summary_table = comparison_df.agg(["mean", "std", "min", "max"])

display(summary_table)

# BoostIn usign authors package (slower?? why??)

In [ ]:
start = time.time()

# Fit BoostIn explainer on your already-trained LightGBM model
boostin_explainer = BoostIn()
boostin_explainer.fit(
    fitted_model,
    X_train.to_numpy(),
    y_train.to_numpy()
)

# Compute influence of every train point on every test point
boostin_scores = boostin_explainer.get_local_influence(
    X_test.to_numpy(),
    y_test.to_numpy()
)

end = time.time()

boostin_df = pd.DataFrame(
    boostin_scores,
    index=X_train.index,
    columns=X_test.index
)

print("BoostIn runtime:", end - start, "seconds")
print("Shape:", boostin_df.shape)

#Compare results
target_idx = X_test.index[0]

loo_top10 = (
    loo_df[target_idx]
    .sort_values(ascending=False)
    .head(10)
    .rename("loo_score")
    .to_frame()
)

boostin_top10 = (
    boostin_df[target_idx]
    .sort_values(ascending=False)
    .head(10)
    .rename("boostin_score")
    .to_frame()
)

print("LOO top supporters:")
display(loo_top10)

print("BoostIn top supporters:")
display(boostin_top10)